# Detección de fraude con PySpark ML

Dataset: `bank_transactions` en BigQuery (transacciones bancarias con la etiqueta `is_suspicious`). Cluster: el de la sección B del README de MLOps.

Tres bloques, del más simple al más útil:

| Bloque | Técnica | Tipo de aprendizaje | Pregunta |
|---|---|---|---|
| **A** | K-Means + PCA | No supervisado | ¿Hay grupos naturales de transacciones? |
| **B** | Random Forest | Supervisado | ¿Podemos predecir el fraude con los datos que tenemos? |
| **C** | Reconstrucción con PCA (autoencoder lineal) | No supervisado (aprende solo de lo normal) | ¿Qué tan raro es cada movimiento? |

Las gráficas usan **matplotlib** (Plotly no funciona en este entorno sin acceso a internet).

In [ ]:
# Parche de compatibilidad para el Jupyter de Dataproc: en este entorno matplotlib llama a
# RcParams._get, que no existe. Debe ejecutarse ANTES de importar pyplot.
import matplotlib
_RcParams = type(matplotlib.rcParams)
if not hasattr(_RcParams, '_get'):
    _RcParams._get = _RcParams.get

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.functions import col, hour, dayofweek
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer, OneHotEncoder, PCA
from pyspark.ml.functions import vector_to_array
from pyspark.ml.clustering import KMeans
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, ClusteringEvaluator

In [ ]:
spark = SparkSession.builder \
    .appName("BigQuery_to_PySpark_ML") \
    .getOrCreate()

In [ ]:
PROJECT_ID = 'big-data-lunes-2026-1'
DATASET_ID = 'bank'
TABLE_ID = 'bank_transactions'
BUCKET_TEMP = 'big-data-lunes-20260223'

## Los datos
Se leen directo de BigQuery con el conector de Spark (viene instalado en Dataproc 2.1).

In [ ]:
%%time
df_bank = spark.read.format("bigquery") \
    .option("table", f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}") \
    .load()

In [ ]:
df_bank.printSchema()

In [ ]:
df_bank.count()

# BLOQUE A — Clustering (segmentación de riesgo)

**Idea:** sin usar la etiqueta de fraude, agrupar transacciones parecidas. Si el fraude cae concentrado en algún grupo, ese grupo merece revisión.

Primero se prepara el dato con un `Pipeline`: `StringIndexer` → `OneHotEncoder` → `VectorAssembler` → `StandardScaler`. El escalado importa: K-Means usa distancias, y sin escalar el `amount` (miles) aplastaría a las variables de 0/1.

In [ ]:
df_base = df_bank.select(
    "amount", "from_country", "from_bank", "transaction_type", "timestamp", "is_suspicious"
).withColumn("hour", hour(col("timestamp"))) \
 .withColumn("day", dayofweek(col("timestamp"))) \
 .withColumn("label_real", col("is_suspicious").cast("double")) \
 .na.drop()

In [ ]:
df_base.show()

In [ ]:
cat_cols = ["from_country", "from_bank", "transaction_type"]

indexers = [StringIndexer(inputCol=c, outputCol=c+"_idx", handleInvalid="skip") for c in cat_cols]

encoder = OneHotEncoder(inputCols=[c+"_idx" for c in cat_cols], outputCols=[c+"_vec" for c in cat_cols])

assembler = VectorAssembler(
    inputCols=["amount", "hour", "day"] + [c+"_vec" for c in cat_cols],
    outputCol="features_raw"
)

scaler = StandardScaler(inputCol="features_raw", outputCol="features_scaled")

In [ ]:
%%time
prep_pipeline = Pipeline(stages=indexers + [encoder, assembler, scaler])
prep_model = prep_pipeline.fit(df_base)
df_final = prep_model.transform(df_base)

In [ ]:
df_final.select("amount", "hour", "day", "label_real", "features_scaled").show(5, truncate=60)

### ¿Cuántos grupos? El método del codo

K-Means necesita que le digamos K. Se entrena con varios valores y se mira el **costo** (suma de distancias al centro de su grupo, `trainingCost`): siempre baja al subir K; buscamos el punto donde deja de bajar rápido (el "codo").

Cada entrenamiento tarda cerca de 15-25 s en el cluster B, por eso `K_MAX` empieza en 6. Súbelo si quieres ver más del panorama.

In [ ]:
%%time
K_MAX = 6          # se prueban K = 2 ... K_MAX-1
cost = []
print(f"Calculando el método del codo para K de 2 a {K_MAX - 1}...")
for k in range(2, K_MAX):
    modelo_k = KMeans(featuresCol="features_scaled", k=k, seed=42).fit(df_final)
    cost.append(modelo_k.summary.trainingCost)
    print(f"  K={k}  costo={cost[-1]:,.0f}")

In [ ]:
# Crear DataFrame para el codo
df_elbow = pd.DataFrame({'K': list(range(2, K_MAX)), 'WSS': cost})

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(df_elbow['K'], df_elbow['WSS'], marker='o', linestyle='-', color='#1f77b4', linewidth=2, markersize=6)
ax.set_title('Método del Codo (Selección de K)', fontsize=14, pad=15, fontweight='bold')
ax.set_xlabel('K', fontsize=12)
ax.set_ylabel('WSS (costo de entrenamiento)', fontsize=12)
ax.grid(True, linestyle='--', alpha=0.5, color='#ccc')
ax.set_xticks(df_elbow['K'])
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.show()

**Lectura de la gráfica:** con estos datos el codo **no es nítido** — el costo baja de forma casi pareja. Cuando pasa esto, K se elige por utilidad (¿cuántos grupos puedo revisar y explicar?), no porque la gráfica lo ordene. Usaremos **K = 4**.

Otra referencia es el coeficiente de silueta (de -1 a 1, más alto = grupos mejor separados). Se calcula sobre una muestra porque es costoso:

In [ ]:
%%time
k_optimo = 4
kmeans_final = KMeans(featuresCol="features_scaled", k=k_optimo, predictionCol="cluster", seed=42)
model_final = kmeans_final.fit(df_final)
df_res = model_final.transform(df_final)

silueta = ClusteringEvaluator(featuresCol="features_scaled", predictionCol="cluster",
                              metricName="silhouette").evaluate(df_res.sample(0.05, seed=42))
print(f"Silueta (muestra 5%) con K={k_optimo}: {silueta:.3f}")

print("\nFraude por cluster (¿se concentra en alguno?):")
(df_res.groupBy("cluster")
       .agg(F.count("*").alias("transacciones"),
            F.round(F.avg("label_real") * 100, 2).alias("pct_fraude"))
       .orderBy("cluster").show())

### Ver los clusters en 2D con PCA

`PCA` comprime las variables a 2 componentes para poder graficarlas. Ojo con `explainedVariance`: indica qué fracción de la información conserva cada componente.

In [ ]:
pca = PCA(k=2, inputCol="features_scaled", outputCol="pca_features")
pca_model = pca.fit(df_res)
df_vis = pca_model.transform(df_res)
print("Varianza explicada por componente:", pca_model.explainedVariance)
print(f"Total en 2D: {sum(pca_model.explainedVariance):.1%}")

Si el total es bajo (con variables categóricas convertidas a 0/1 suele serlo), el gráfico 2D es una **proyección pobre**: dos puntos que se ven juntos pueden estar lejos en el espacio real. Sirve para intuición, no como prueba de que los grupos están bien separados (para eso está la silueta).

In [ ]:
# --- CONFIGURACIÓN DE SEGURIDAD ---
MAX_RECORDS_DRIVER = 15000  # Límite seguro para no saturar la RAM del Driver
print("Iniciando proceso de visualización segura...")

# 1. Calcular el conteo total de registros disponibles tras PCA
total_count = df_vis.count()
print(f"Total de registros en df_vis: {total_count}")

# 2. Determinar la fracción de muestreo óptima
sampling_fraction = min(1.0, MAX_RECORDS_DRIVER / total_count)

if sampling_fraction < 1.0:
    print(f"El dataset es grande. Muestreando al {sampling_fraction*100:.2f}% para seguridad.")
else:
    print("Dataset manejable. Procediendo con todos los datos seleccionados.")

# 3. Ejecutar el SELECT y el muestreo antes de toPandas()
df_to_convert = df_vis.select(
    "pca_features", "cluster", "label_real", "amount", "from_bank"
).sample(withReplacement=False, fraction=sampling_fraction, seed=42)

# Ahora es seguro convertir
sample_pd = df_to_convert.toPandas()

# 4. PROCESAMIENTO EN PANDAS
coords = pd.DataFrame(
    sample_pd['pca_features'].apply(lambda x: x.toArray()).tolist(), 
    columns=['x', 'y']
)
sample_pd = pd.concat([sample_pd, coords], axis=1)

# Crear etiqueta legible para el gráfico
sample_pd['Tipo'] = sample_pd['label_real'].apply(lambda x: 'ALERTA: FRAUDE' if x == 1.0 else 'Normal')

# 5. GRÁFICO SCATTER (Matplotlib con paleta "Safe" equivalente)
# Forzamos un fondo oscuro similar a 'plotly_dark' de forma manual y robusta
fig, ax = plt.subplots(figsize=(12, 8), facecolor='#111111')
ax.set_facecolor('#1e1e1e')

# Paleta de colores cualitativa segura (equivalente a px.colors.qualitative.Safe)
colores_safe = ['#4477AA', '#EE6677', '#228833', '#CCBB44', '#66CCEE', '#AA3377', '#BBBBBB']
clusters_unicos = sorted(sample_pd['cluster'].unique())
color_map = {c: colores_safe[i % len(colores_safe)] for i, c in enumerate(clusters_unicos)}

# Mapeo de marcadores según el 'Tipo' (Círculo para Normal, 'X' grande para Fraude)
marcadores_map = {'Normal': {'marker': 'o', 'alpha': 0.6, 'size': 30},
                  'ALERTA: FRAUDE': {'marker': 'x', 'alpha': 1.0, 'size': 100}}

# Graficar iterando por tipo y cluster para generar la leyenda correctamente
for tipo, config in marcadores_map.items():
    df_tipo = sample_pd[sample_pd['Tipo'] == tipo]
    
    for cluster in clusters_unicos:
        df_sub = df_tipo[df_tipo['cluster'] == cluster]
        
        if not df_sub.empty:
            # Si es fraude, fijamos un color llamativo (ej. Rojo) o seguimos el mapa de clusters
            color_punto = '#FF3333' if tipo == 'ALERTA: FRAUDE' else color_map[cluster]
            
            ax.scatter(
                df_sub['x'], df_sub['y'],
                c=color_punto,
                marker=config['marker'],
                s=config['size'],
                alpha=config['alpha'],
                label=f"Cluster {cluster} ({tipo})" if tipo == 'ALERTA: FRAUDE' else f"Cluster {cluster}"
            )

# Título y etiquetas con estilos claros sobre fondo oscuro
ax.set_title(f"Clusters de Transacciones y Fraude (K={k_optimo}) - Muestra Segura", 
             color='white', fontsize=14, pad=20, fontweight='bold')
ax.set_xlabel('Componente Principal X', color='#aaaaaa', fontsize=12)
ax.set_ylabel('Componente Principal Y', color='#aaaaaa', fontsize=12)

# Configurar cuadrícula sutil y ejes
ax.grid(True, linestyle=':', alpha=0.3, color='#555555')
ax.tick_params(colors='#aaaaaa', labelsize=10)

# Remover los bordes de la gráfica para un look moderno
for spine in ax.spines.values():
    spine.set_visible(False)

# Construir la leyenda de forma limpia evitando duplicados gigantes
handles, labels = ax.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
ax.legend(by_label.values(), by_label.keys(), 
          facecolor='#2e2e2e', edgecolor='none', labelcolor='white', loc='upper right')

# Desplegar en el Notebook de Apache Spark sin requerir internet
plt.show()

# BLOQUE B — Clasificación (predicción de fraude)

**Idea:** ahora sí usamos la etiqueta. Un `RandomForestClassifier` aprende, a partir de ejemplos, qué transacciones fueron sospechosas.

Reutiliza **exactamente el mismo preprocesamiento del Bloque A** (índices + one-hot + ensamblado, sin escalar: los árboles no lo necesitan) y añade:
- **Desbalance:** el fraude es una fracción pequeña. Un modelo que dijera siempre "normal" ya acertaría ~97%, así que la exactitud no sirve. Se pondera cada clase inversamente a su frecuencia (`weightCol`) y se evalúa con **AUC-PR**, además de AUC-ROC.
- **Split antes de ajustar:** el modelo nunca ve el conjunto de prueba.

In [ ]:
%%time
df_clf = df_base.withColumn("label", col("label_real"))

n = df_clf.count()
n_pos = df_clf.filter(col("label") == 1).count()
n_neg = n - n_pos
print(f"Fraude: {n_pos:,} de {n:,} ({n_pos / n:.2%}). Un modelo que siempre diga 'normal' acierta {n_neg / n:.2%}.")

df_clf = df_clf.withColumn("peso", F.when(col("label") == 1, n / (2.0 * n_pos)).otherwise(n / (2.0 * n_neg)))

train, test = df_clf.randomSplit([0.8, 0.2], seed=42)

rf = RandomForestClassifier(labelCol="label", featuresCol="features_raw", weightCol="peso",
                            numTrees=50, seed=42)
pipe_rf = Pipeline(stages=indexers + [encoder, assembler, rf])   # mismas etapas del Bloque A
model_rf = pipe_rf.fit(train)
preds = model_rf.transform(test)

auc_roc = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC").evaluate(preds)
auc_pr  = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderPR").evaluate(preds)
print(f"\n>>> AUC-ROC = {auc_roc:.4f} | AUC-PR = {auc_pr:.4f} (un modelo al azar tiene AUC-ROC 0.5 y AUC-PR ≈ {n_pos / n:.3f}) <<<")

imp = model_rf.stages[-1].featureImportances.toArray()
print("\nImportancia de las 3 variables numéricas:",
      {"amount": round(float(imp[0]), 4), "hour": round(float(imp[1]), 4), "day": round(float(imp[2]), 4)})
print("Las demás posiciones son las categorías (país, banco, tipo) ya convertidas a 0/1.")

**Cómo leer este resultado:**
- **AUC-ROC ≈ 0.5** significa que el modelo no distingue fraude de normal con estas columnas. **No es un error del código:** es información — la señal del fraude no está en el monto, la hora, el día, el país, el banco ni el tipo de transacción.
- Si es el caso, el camino es **buscar variables nuevas** (`df_bank.columns` muestra todas: por ejemplo, datos del destinatario o el historial del cliente), no cambiar de algoritmo.
- ⚠️ **Cuidado con la fuga de información:** si existe una columna que describe *por qué* la transacción se marcó como sospechosa (como `suspicious_pattern`), usarla como variable daría un AUC casi perfecto sin servir en la vida real, porque solo se conoce **después** de etiquetar.

# BLOQUE C — Detección de anomalías por error de reconstrucción

**Idea:** el Random Forest necesita ejemplos de fraude para aprender. Pero los defraudadores cambian de táctica, y casi siempre hay muchísimos más ejemplos de lo normal que de fraude. Otra forma de pensarlo: **aprender cómo se ve lo normal y marcar lo que no se parece.**

Se hace con un **autoencoder**: una red que comprime cada transacción a pocos números (el *cuello de botella*) e intenta reconstruirla. Si aprendió solo de transacciones normales, reconstruye bien lo normal y **mal lo raro** → el *error de reconstrucción* es el puntaje de anomalía.

> **Spark MLlib no trae autoencoders.** Lo que sí trae es `PCA`, que es matemáticamente un **autoencoder lineal**: el "encoder" proyecta a `k` componentes y el "decoder" regresa al espacio original. Es honesto y suficiente para ver la idea completa. Para uno no lineal (con capas ocultas) se usaría PyTorch o Keras — el cluster B.1 ya trae `torch` instalado.

Reglas para hacerlo bien (y que las métricas sean confiables):
1. **Split antes de ajustar nada**, y el preprocesamiento se ajusta solo con train.
2. El modelo aprende **solo de transacciones normales de train** (no ve el fraude al aprender).
3. El umbral se define con datos de train, no con el conjunto que se evalúa.

In [ ]:
%%time
# 1. Split y preprocesamiento SOLO con train (centrado + escalado: PCA necesita datos centrados)
train_c, test_c = df_base.randomSplit([0.8, 0.2], seed=42)

scaler_c = StandardScaler(inputCol="features_raw", outputCol="features_scaled", withMean=True, withStd=True)
prep_c = Pipeline(stages=indexers + [encoder, assembler, scaler_c]).fit(train_c)
train_f = prep_c.transform(train_c)
test_f = prep_c.transform(test_c)

# 2. El "autoencoder": PCA con K_BOTTLENECK componentes, ajustado SOLO con transacciones normales de train
K_BOTTLENECK = 8
normales_train = train_f.filter(col("label_real") == 0)
ae = PCA(k=K_BOTTLENECK, inputCol="features_scaled", outputCol="codigo").fit(normales_train)
print(f"Variables de entrada: {len(train_f.select('features_scaled').first()[0])} -> cuello de botella: {K_BOTTLENECK}")
print(f"Información que conserva el cuello de botella: {sum(ae.explainedVariance):.1%}")

# 3. Error de reconstrucción = distancia entre la transacción y su reconstrucción.
#    Como las columnas de PCA son ortonormales: ||x - x_reconstruida||^2 = ||x||^2 - ||codigo||^2
def norma2(c):
    return F.aggregate(vector_to_array(c), F.lit(0.0), lambda acc, x: acc + x * x)

def puntuar(d):
    d = ae.transform(d)
    return d.withColumn("anomaly_score", F.sqrt(F.greatest(norma2("features_scaled") - norma2("codigo"), F.lit(0.0))))

train_scored = puntuar(train_f)
df_predictions = puntuar(test_f).persist()

In [ ]:
# 4. Umbral: percentil 95 del error en transacciones NORMALES de TRAIN
umbral = train_scored.filter(col("label_real") == 0).approxQuantile("anomaly_score", [0.95], 0.001)[0]
print(f"Umbral (percentil 95 de normales en train): {umbral:.4f}")

# 5. Evaluación sobre TODO el conjunto de prueba, calculada en Spark (no en una muestra)
df_predictions = df_predictions.withColumn("alerta", (col("anomaly_score") > umbral).cast("int"))
print("\nMatriz de confusión (filas = realidad, columnas = alerta):")
df_predictions.groupBy("label_real").pivot("alerta", [0, 1]).count().na.fill(0).orderBy("label_real").show()

cm = {(r["label_real"], r["alerta"]): r["count"] for r in df_predictions.groupBy("label_real", "alerta").count().collect()}
tp, fn, fp = cm.get((1.0, 1), 0), cm.get((1.0, 0), 0), cm.get((0.0, 1), 0)
print(f"Fraude detectado (recall): {tp / max(tp + fn, 1):.1%} | de las alertas, fraude real (precisión): {tp / max(tp + fp, 1):.1%}")

roc_ae = BinaryClassificationEvaluator(labelCol="label_real", rawPredictionCol="anomaly_score", metricName="areaUnderROC").evaluate(df_predictions)
pr_ae = BinaryClassificationEvaluator(labelCol="label_real", rawPredictionCol="anomaly_score", metricName="areaUnderPR").evaluate(df_predictions)
print(f"AUC-ROC = {roc_ae:.4f} | AUC-PR = {pr_ae:.4f}")

**Puntos clave para la clase:**
1. **El cuello de botella obliga a resumir.** Con menos números que variables de entrada, el modelo solo puede conservar lo más común; lo raro no cabe y se reconstruye mal.
2. **El umbral es una decisión de negocio.** El percentil 95 significa aceptar revisar ~5% de lo normal como falsa alarma. Subirlo o bajarlo cambia cuántos fraudes se detectan y cuánto cuesta revisarlos: prueba con 0.90 y 0.99.
3. **Contraste con el Random Forest:** él necesita que le digan qué es fraude; este solo necesita lo normal. Por eso sirve contra tácticas nuevas, aunque casi siempre detecta menos que un modelo supervisado que sí encontró señal.

Ahora las gráficas, sobre una **muestra** del conjunto de prueba (`toPandas()` trae todo al driver; nunca se hace con 460 mil filas).

In [ ]:
from sklearn.metrics import precision_recall_curve, auc as sk_auc

MAX_ROWS = 10000
total_rows = df_predictions.count()
print(f"Filas de prueba: {total_rows:,}")
if total_rows > MAX_ROWS:
    print(f"Tomando una muestra de ~{MAX_ROWS:,} filas para proteger la memoria del Driver...")
    df_to_plot = df_predictions.sample(withReplacement=False, fraction=MAX_ROWS / total_rows, seed=42)
else:
    df_to_plot = df_predictions

sample_pd = df_to_plot.select("amount", "label_real", "anomaly_score").toPandas()

In [ ]:
# Separación de las clases en Pandas para los gráficos
normales = sample_pd[sample_pd['label_real'] == 0]
fraudulentos = sample_pd[sample_pd['label_real'] == 1]

# =====================================================================
# 2. GRÁFICO 1: DETECCIÓN DE ANOMALÍAS (SCATTER ERROR DE RECONSTRUCCIÓN)
# =====================================================================
fig1, ax1 = plt.subplots(figsize=(12, 5), facecolor='#111111')
ax1.set_facecolor('#1e1e1e')

# Capa 1: Normales
ax1.scatter(
    normales['amount'], normales['anomaly_score'],
    color='lightgrey', s=36, alpha=0.3, edgecolors='none', label='Normal (Bajo Riesgo)'
)
# Capa 2: Fraudulentos
ax1.scatter(
    fraudulentos['amount'], fraudulentos['anomaly_score'],
    color='red', s=144, alpha=1.0, marker='x', linewidths=2, edgecolors='darkred', label='ALERTA: FRAUDE REAL'
)

ax1.set_title("Detección de Anomalías: Error de Reconstrucción (PCA como autoencoder lineal)", color='white', fontsize=13, pad=15, fontweight='bold')
ax1.set_xlabel("Monto de la Transacción", color='#aaaaaa', fontsize=11)
ax1.set_ylabel("Error de reconstrucción", color='#aaaaaa', fontsize=11)
ax1.grid(True, linestyle=':', alpha=0.3, color='#555555')
ax1.tick_params(colors='#aaaaaa')
for spine in ax1.spines.values(): spine.set_visible(False)
ax1.legend(loc="upper left", facecolor=(0,0,0,0.5), edgecolor='none', labelcolor='white')
plt.tight_layout()
plt.show()

# =====================================================================
# 3. GRÁFICO 2: DENSIDAD (DISTPLOT EQUIVALENTE)
# =====================================================================
fig2, ax2 = plt.subplots(figsize=(12, 4), facecolor='#111111')
ax2.set_facecolor('#1e1e1e')

# Histogramas normados para emular la densidad de distribución de Plotly
bins = np.linspace(sample_pd['anomaly_score'].min(), sample_pd['anomaly_score'].max(), 50)
ax2.hist(normales['anomaly_score'], bins=bins, density=True, alpha=0.5, color='#3498db', label='Normal')
ax2.hist(fraudulentos['anomaly_score'], bins=bins, density=True, alpha=0.5, color='#e74c3c', label='Fraude')

ax2.set_title("¿Funciona? Separación de errores entre clases", color='white', fontsize=13, pad=15, fontweight='bold')
ax2.set_xlabel("Error de reconstrucción", color='#aaaaaa', fontsize=11)
ax2.set_ylabel("Densidad", color='#aaaaaa', fontsize=11)
ax2.grid(True, linestyle=':', alpha=0.3, color='#555555')
ax2.tick_params(colors='#aaaaaa')
for spine in ax2.spines.values(): spine.set_visible(False)
ax2.legend(loc="upper right", facecolor=(0,0,0,0.5), edgecolor='none', labelcolor='white')
plt.tight_layout()
plt.show()

# =====================================================================
# 4. MATRIZ DE CONFUSIÓN Y GRÁFICO 3 (HEATMAP)
# =====================================================================
print(f"Umbral (calculado en train): {umbral:.4f}")

sample_pd['prediccion_anomalia'] = (sample_pd['anomaly_score'] > umbral).astype(int)
matriz = pd.crosstab(
    sample_pd['label_real'],
    sample_pd['prediccion_anomalia'],
    rownames=['Realidad'],
    colnames=['Alerta']
).reindex(index=[0.0, 1.0], columns=[0, 1], fill_value=0)
print("\n--- Matriz de Confusión (solo la muestra graficada) ---")
print(matriz)

# Mapa de calor de la Matriz
fig3, ax3 = plt.subplots(figsize=(6, 4.5))
cax = ax3.imshow(matriz.values, cmap='YlOrRd', interpolation='nearest')
fig3.colorbar(cax)

ax3.set_title("Matriz de Confusión: Detección por Umbral", fontsize=12, pad=15, fontweight='bold')
ax3.set_xticks(np.arange(len(matriz.columns)))
ax3.set_yticks(np.arange(len(matriz.index)))
ax3.set_xticklabels(matriz.columns)
ax3.set_yticklabels(matriz.index)
ax3.set_xlabel('Alerta', fontsize=10)
ax3.set_ylabel('Realidad', fontsize=10)

# Escribir los números dentro de los cuadrantes
for i in range(len(matriz.index)):
    for j in range(len(matriz.columns)):
        ax3.text(j, i, str(matriz.iloc[i, j]), ha='center', va='center', 
                 color='black' if matriz.iloc[i, j] < (matriz.values.max()/2) else 'white',
                 fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

# =====================================================================
# 5. GRÁFICO 4: CURVA PRECISION-RECALL (ÁREA EQUIVALENTE)
# =====================================================================
precision, recall, _ = precision_recall_curve(sample_pd['label_real'], sample_pd['anomaly_score'])
area_bajo_curva = sk_auc(recall, precision)

fig4, ax4 = plt.subplots(figsize=(12, 5), facecolor='#111111')
ax4.set_facecolor('#1e1e1e')

# Rellenar el área bajo la curva (emula px.area)
ax4.fill_between(recall, precision, color='#3498db', alpha=0.4)
ax4.plot(recall, precision, color='#3498db', linewidth=2.5, label='Curva PR')

# Línea base aleatoria (dash) equivalente al layout original de Plotly
prevalencia = (sample_pd['label_real'] == 1).mean()
ax4.axhline(y=prevalencia, color='#e74c3c', linestyle='--', linewidth=1.5, label=f'Línea base (azar) = {prevalencia:.3f}')

ax4.set_title(f"Curva Precision-Recall (AUC: {area_bajo_curva:.4f})", color='white', fontsize=13, pad=15, fontweight='bold')
ax4.set_xlabel("Recall (Sensibilidad)", color='#aaaaaa', fontsize=11)
ax4.set_ylabel("Precision (Precisión)", color='#aaaaaa', fontsize=11)
ax4.grid(True, linestyle=':', alpha=0.3, color='#555555')
ax4.tick_params(colors='#aaaaaa')
ax4.set_xlim([0.0, 1.0])
ax4.set_ylim([0.0, 1.05])
for spine in ax4.spines.values(): spine.set_visible(False)
ax4.legend(loc="lower left", facecolor=(0,0,0,0.5), edgecolor='none', labelcolor='white')

plt.tight_layout()
plt.show()

## Para practicar
1. **Umbral:** cambia el percentil (0.90, 0.99) y observa cómo se mueven el recall y la precisión de la celda de evaluación. ¿Cuál elegirías si cada revisión manual cuesta dinero?
2. **Cuello de botella:** prueba `K_BOTTLENECK` = 2, 4, 12. ¿Qué pasa con el AUC-PR al comprimir de más o de menos?
3. **Variables nuevas:** en el Bloque B, agrega columnas de `df_bank.columns` y observa si el AUC-ROC sale de 0.5.
4. **No lineal:** ¿qué ganaría un autoencoder con capas ocultas (PyTorch) frente a PCA? Discútelo antes de probarlo.